In [ ]:
import os
import glob
import librosa
import numpy as np
import torch
from datasets import Dataset, Audio, ClassLabel
from transformers import (
    ASTFeatureExtractor,
    ASTForAudioClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model


In [ ]:
#!unzip patient-vocal-dataset.zip

In [ ]:

# --- 1. Configuration ---
ROOT_AUDIO_DIR = "patient-vocal-dataset"
LABELS = ["Laryngozele", "Normal", "Vox senilis"]
MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"


In [ ]:

# --- 2. Load Data from Folders ---
def create_dataset_from_folders(root_dir, labels):
    data = []
    print(f"Scanning {root_dir}...")

    for label in labels:
        folder_path = os.path.join(root_dir, label)
        # Check if folder exists
        if not os.path.exists(folder_path):
            print(f"Warning: Folder '{folder_path}' not found.")
            continue

        # Get all wav files
        wav_files = glob.glob(os.path.join(folder_path, "*.wav"))
        print(f"Found {len(wav_files)} files in '{label}'")

        for file_path in wav_files:
            data.append({"audio": file_path, "label": label})

    return Dataset.from_list(data)

# Create the raw dataset
raw_dataset = create_dataset_from_folders(ROOT_AUDIO_DIR, LABELS)

# Create ClassLabel feature to handle string-to-id mapping automatically
class_label = ClassLabel(names=LABELS)
raw_dataset = raw_dataset.cast_column("label", class_label)

# Split into Train and Test (80% Train, 20% Test) so you know if it works
dataset = raw_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)


Scanning patient-vocal-dataset...
Found 84 files in 'Laryngozele'
Found 560 files in 'Normal'
Found 392 files in 'Vox senilis'


Casting the dataset:   0%|          | 0/1036 [00:00<?, ? examples/s]

In [ ]:

# --- 3. Preprocessing (Audio -> Spectrogram) ---
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    # 1. Load audio arrays using librosa
    # AST requires 16kHz sampling rate
    audio_arrays = [librosa.load(x, sr=16000)[0] for x in examples["audio"]]

    # 2. Convert to Spectrograms (inputs)
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=16000,
        padding="max_length",
        return_tensors="pt"
    )

    # 3. Add labels
    inputs["labels"] = examples["label"]
    return inputs

print("Preprocessing audio (this might take a moment)...")
encoded_dataset = dataset.map(preprocess_function, batched=True, batch_size=10)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Preprocessing audio (this might take a moment)...


Map:   0%|          | 0/828 [00:00<?, ? examples/s]

Map:   0%|          | 0/208 [00:00<?, ? examples/s]

In [ ]:

# --- 4. Model Setup ---
# Map labels to IDs (e.g., "Normal" -> 1)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for i, label in enumerate(LABELS)}

model = ASTForAudioClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True # Essential when changing from ImageNet classes to your 3 classes
)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                       
------------------------+----------+---------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527, 768]) vs model:torch.Size([3, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([527]) vs model:torch.Size([3])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [ ]:

# --- 5. Metrics ---
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }


In [ ]:

# --- 6. Training ---
training_args = TrainingArguments(
    output_dir="./audio_classification_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    fp16=True,                       # Use False if not on a CUDA GPU
    remove_unused_columns=False      # Keep 'audio' column for reference if needed
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    #tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:

# Start Training
trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.502648,0.434323,0.841346,0.809550,0.782330,0.841346
2,0.298276,0.412737,0.884615,0.874039,0.889599,0.884615
3,0.124239,0.495723,0.855769,0.857591,0.871780,0.855769
4,0.001556,0.333326,0.932692,0.932015,0.933286,0.932692
5,0.000424,0.337049,0.927885,0.927857,0.928309,0.927885


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=520, training_loss=0.25054373021882315, metrics={'train_runtime': 452.3842, 'train_samples_per_second': 9.152, 'train_steps_per_second': 1.149, 'total_flos': 2.8062346089332736e+17, 'train_loss': 0.25054373021882315, 'epoch': 5.0})

In [ ]:

# Save the final model
trainer.save_model("./stage2_laryngeal_model")
feature_extractor.save_pretrained("./stage2_laryngeal_model")

print("Training complete. Model saved to ./stage2_laryngeal_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Model saved to ./stage2_laryngeal_model


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy
from transformers import (
    ASTForAudioClassification,
    ASTFeatureExtractor,
    TrainingArguments,
    Trainer
)
from datasets import load_dataset, Dataset, Audio
import glob
import os
import librosa


In [ ]:

# --- Configuration ---
# Point this to the model you trained in the previous step
DENSE_MODEL_PATH = "stage2_laryngeal_model"
NEW_MOE_MODEL_NAME = "ast-voice-moe-stage3"
ROOT_AUDIO_DIR = "patient-vocal-dataset"
LABELS = ["Laryngozele", "Normal", "Vox senilis"]

NUM_EXPERTS = 4
TOP_K = 2
AUX_LOSS_COEF = 0.01


In [ ]:
###V1
class ASTFFNExpert(nn.Module):
    def __init__(self, original_intermediate, original_output):
        super().__init__()
        self.intermediate = copy.deepcopy(original_intermediate)
        self.output = copy.deepcopy(original_output)

    def forward(self, hidden_states):
        inter = self.intermediate(hidden_states)
        return self.output(inter, hidden_states)
class FFNIdentity(nn.Module):
    def forward(self, hidden_states, input_tensor=None):
        # AST expects output(intermediate, residual)
        return hidden_states

class ASTMoEFFN(nn.Module):
    def __init__(self, original_intermediate, original_output,
                 num_experts=4, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        hidden_dim = original_intermediate.dense.in_features  # 768
        self.router = nn.Linear(hidden_dim, num_experts, bias=False)

        self.experts = nn.ModuleList([
            ASTFFNExpert(original_intermediate, original_output)
            for _ in range(num_experts)
        ])

        self.router_logits = None
        self.prev_logits = None

    def forward(self, hidden_states):
        B, T, D = hidden_states.shape
        flat_x = hidden_states.reshape(-1, D)

        # ---- routing on 768-dim tokens ----
        logits = self.router(flat_x)
        self.router_logits = logits

        self.router_logits = logits
        probs = F.softmax(logits, dim=-1)

        topk_vals, topk_idx = probs.topk(self.top_k, dim=-1)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(flat_x)

        for k in range(self.top_k):
            expert_ids = topk_idx[:, k]
            weights = topk_vals[:, k].unsqueeze(-1)

            for eid, expert in enumerate(self.experts):
                mask = (expert_ids == eid)
                if not mask.any():
                    continue

                y = expert(flat_x[mask].unsqueeze(1)).squeeze(1)
                output[mask] += y * weights[mask]

        return output.view(B, T, D)


In [ ]:
###V2
class ASTFFNExpert(nn.Module):
    def __init__(self, original_intermediate, original_output):
        super().__init__()
        # 1. First Dense Layer (Upscale) + Activation
        self.dense_in = copy.deepcopy(original_intermediate.dense)
        self.act = original_intermediate.intermediate_act_fn

        # 2. Second Dense Layer (Downscale) + Dropout
        # We handle the LayerNorm OUTSIDE the expert now to avoid "Double Norm"
        self.dense_out = copy.deepcopy(original_output.dense)
        self.dropout = copy.deepcopy(original_output.dropout)

    def forward(self, hidden_states):
        # Pure FFN transformation: f(x)
        x = self.dense_in(hidden_states)
        x = self.act(x)
        x = self.dense_out(x)
        x = self.dropout(x)
        return x

class FFNIdentity(nn.Module):
    def forward(self, hidden_states, input_tensor=None):
        # The ASTLayer passes (expert_output, residual_input) to this module.
        # We must add them together to preserve the ResNet-like structure.
        return hidden_states + input_tensor

class ASTMoEFFN(nn.Module):
    def __init__(self, original_intermediate, original_output,
                 num_experts=4, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k

        hidden_dim = original_intermediate.dense.in_features
        self.router = nn.Linear(hidden_dim, num_experts, bias=False)

        # Experts contain the FC1 -> Act -> FC2 -> Dropout logic
        self.experts = nn.ModuleList([
            ASTFFNExpert(original_intermediate, original_output)
            for _ in range(num_experts)
        ])

        # We do NOT need to copy LayerNorm here.
        # In AST, LayerNorm is handled by the parent ASTLayer (self.layernorm_after).

        self.router_logits = None

    def forward(self, hidden_states):
        B, T, D = hidden_states.shape
        flat_x = hidden_states.reshape(-1, D)

        # ---- Routing ----
        logits = self.router(flat_x)
        self.router_logits = logits

        probs = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = probs.topk(self.top_k, dim=-1)
        topk_vals = topk_vals / topk_vals.sum(dim=-1, keepdim=True)

        expert_output = torch.zeros_like(flat_x)

        for k in range(self.top_k):
            expert_ids = topk_idx[:, k]
            weights = topk_vals[:, k].unsqueeze(-1)

            for eid, expert in enumerate(self.experts):
                mask = (expert_ids == eid)
                if not mask.any():
                    continue

                # Experts process raw token (flat_x)
                y = expert(flat_x[mask].unsqueeze(1)).squeeze(1)
                expert_output[mask] += y * weights[mask]

        expert_output = expert_output.view(B, T, D)

        # Return the delta. The residual addition happens in FFNIdentity.
        return expert_output

In [ ]:

# --- 2. Custom MoE Trainer ---

class AudioMoETrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Forward pass
        outputs = model(**inputs)
        ce_loss = outputs.loss

        # Calculate Aux Loss (Load Balancing)
        aux_loss = 0.0
        num_experts = NUM_EXPERTS
        encoder_layers = model.audio_spectrogram_transformer.encoder.layer

        for layer in encoder_layers:
            if isinstance(layer.intermediate, ASTMoEFFN):
                logits = layer.intermediate.router_logits
                if logits is None:
                    continue

                probs = F.softmax(logits, dim=-1)          # p_i
                mean_probs = probs.mean(0)

                # token → expert frequency f_i
                top1 = probs.argmax(dim=-1)
                counts = torch.bincount(top1, minlength=num_experts).float()
                freq = counts / counts.sum()

                aux_loss += num_experts * torch.sum(mean_probs * freq)

        total_loss = ce_loss + (AUX_LOSS_COEF * aux_loss)
        return (total_loss, outputs) if return_outputs else total_loss


In [ ]:

# --- 3. Main Execution ---

# A. Load Model
print(f"Loading Dense Model from {DENSE_MODEL_PATH}...")
model = ASTForAudioClassification.from_pretrained(DENSE_MODEL_PATH)
feature_extractor = ASTFeatureExtractor.from_pretrained(DENSE_MODEL_PATH)

# B. Convert to MoE
print("Converting AST layers to Mixture of Experts...")

# AST Encoder Layers
layers = model.audio_spectrogram_transformer.encoder.layer
num_layers = len(layers)

for i, layer in enumerate(layers):
    if i < num_layers // 2:
        continue

    moe_ffn = ASTMoEFFN(
        layer.intermediate,
        layer.output,
        num_experts=NUM_EXPERTS,
        top_k=TOP_K
    )

    layer.intermediate = moe_ffn
    layer.output = FFNIdentity()  # output is now inside MoE


print(f"Converted {len(layers)} layers to MoE.")

# C. Freeze Parameters (Train only Routers + Experts)
for name, param in model.named_parameters():
    if "router" in name or "experts" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {trainable:,}")


Loading Dense Model from stage2_laryngeal_model...


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Converting AST layers to Mixture of Experts...
Converted 12 layers to MoE.
Trainable Parameters: 113,356,800


In [ ]:

# D. Load Dataset
def create_dataset(root_dir, labels):
    data = []
    for label in labels:
        folder_path = os.path.join(root_dir, label)
        for f in glob.glob(os.path.join(folder_path, "*.wav")):
            data.append({"audio": f, "label": label})
    return Dataset.from_list(data)

raw_dataset = create_dataset(ROOT_AUDIO_DIR, LABELS)
label2id = {l: i for i, l in enumerate(LABELS)}
raw_dataset = raw_dataset.class_encode_column("label") # Ensure labels are ints

def preprocess(examples):
    audio_arrays = [librosa.load(x, sr=16000)[0] for x in examples["audio"]]
    inputs = feature_extractor(audio_arrays, sampling_rate=16000, padding="max_length", return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

processed_dataset = raw_dataset.map(preprocess, batched=True, batch_size=4)
split_dataset = processed_dataset.train_test_split(test_size=0.2)


Casting to class labels:   0%|          | 0/1036 [00:00<?, ? examples/s]

Map:   0%|          | 0/1036 [00:00<?, ? examples/s]

In [ ]:
# --- 1. Define the Metrics Function ---
def compute_metrics(eval_pred):
    # eval_pred contains (logits, labels)
    predictions, labels = eval_pred
    # Take the class with the highest score
    predictions = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:

# E. Train
training_args = TrainingArguments(
    output_dir="./results_audio_moe",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    fp16=True,
    remove_unused_columns=False      # Keep 'audio' column for reference if needed

)

trainer = AudioMoETrainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    #tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:

print("Starting MoE Sparsification Training...")
trainer.train()


Starting MoE Sparsification Training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.127538,0.421088,0.913462,0.913188,0.915537,0.913462
2,0.249140,0.466597,0.923077,0.923005,0.924096,0.923077
3,0.123467,0.455300,0.932692,0.932196,0.932422,0.932692
4,0.123291,0.494838,0.918269,0.918478,0.919835,0.918269
5,0.116085,0.521650,0.918269,0.918478,0.919835,0.918269
6,0.122155,0.526942,0.927885,0.927696,0.928421,0.927885
7,0.121768,0.525159,0.927885,0.927696,0.928421,0.927885
8,0.121695,0.527411,0.927885,0.927696,0.928421,0.927885
9,0.121591,0.528485,0.923077,0.922371,0.923393,0.923077
10,0.115580,0.533225,0.918269,0.917680,0.919032,0.918269


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1040, training_loss=0.13071598800329062, metrics={'train_runtime': 1087.2067, 'train_samples_per_second': 7.616, 'train_steps_per_second': 0.957, 'total_flos': 1.1148823742644224e+18, 'train_loss': 0.13071598800329062, 'epoch': 10.0})

In [ ]:

# Save
model.save_pretrained(NEW_MOE_MODEL_NAME)
print(f"MoE Audio Model saved to {NEW_MOE_MODEL_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

MoE Audio Model saved to ast-voice-moe-stage3
